# Import Core Libraries
Load NumPy and pandas for numerical work and dataframe handling.

In [31]:
import numpy as np
import pandas as pd


# Import Machine Learning Tools
Bring in the scikit-learn utilities needed for splitting data, preprocessing, feature selection, modeling, and building the pipeline.

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectKBest , chi2


# Load the Titanic Dataset
Read the training data from the datasets folder and preview a few random rows.

In [33]:
df = pd.read_csv("../../datasets/train.csv")
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
728,729,0,2,"Bryhl, Mr. Kurt Arnold Gottfrid",male,25.0,1,0,236853,26.0000,NaN,S
409,410,0,3,"Lefebre, Miss. Ida",female,NaN,3,1,4133,25.4667,NaN,S
129,130,0,3,"Ekstrom, Mr. Johan",male,45.0,0,0,347061,6.9750,NaN,S
539,540,1,1,"Frolicher, Miss. Hedwig Margaritha",female,22.0,0,2,13568,49.5000,B39,C
473,474,1,2,"Jerwan, Mrs. Amin S (Marie Marthe Thuillard)",female,23.0,0,0,SC/AH Basle 541,13.7917,D,C


# Remove Unused Columns
Drop identifier and high-cardinality text columns that are not used in this simple pipeline.

In [34]:

df = df.drop(columns = ["Cabin", "PassengerId" , "Name" , "Ticket"])

# Check Missing Values
Inspect missing values so the preprocessing steps can handle them correctly.

In [35]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

# Split Features and Target
Separate the input features from the survival target and create training and test sets.

In [36]:
X_train , X_test , Y_train , Y_test = train_test_split(df.drop(columns = ["Survived"]) , df["Survived"] , test_size = 0.2 , random_state = 42)

# Preview Training Features
Look at a sample of the feature data that will be passed into the pipeline.

In [37]:
X_train.sample(5)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
449,1,male,52.0,0,0,30.5000,S
586,2,male,47.0,0,0,15.0000,S
404,3,female,20.0,0,0,8.6625,S
870,3,male,26.0,0,0,7.8958,S
737,1,male,35.0,0,0,512.3292,C


# Preview Training Labels
Check a sample of the target values used for training.

In [38]:
Y_train.sample(5)

128    1
744    1
228    0
440    1
875    1
Name: Survived, dtype: int64

# Review Feature Columns
Confirm the remaining column order before applying column-based transformations.

In [39]:
df.columns

Index(['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
       'Embarked'],
      dtype='object')

# Impute Missing Values
Fill missing Age values and replace missing Embarked values with the most frequent category.

In [40]:
transformer1 = ColumnTransformer(transformers=[
    ("impute_age" , SimpleImputer() , [2]),
    ("impute_embarked" , SimpleImputer(strategy= "most_frequent") , [6])
]
,remainder= "passthrough"
)

# Encode Categorical Features
Apply one-hot encoding to categorical columns while keeping the remaining features for later steps.

In [41]:
transformer2 = ColumnTransformer(transformers=[
    ("ohe_sex_embarked", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), [1, 6])
], remainder="passthrough")

# Scale Numeric Features
Normalize the transformed features so they are on a comparable range for chi-square feature selection.

In [42]:
transformer3 = ColumnTransformer(transformers = [
    ("scaling" , MinMaxScaler()  , slice(0,10))
])

# Select Best Features
Keep the most useful features according to the chi-square score.

In [43]:
transformer4 = SelectKBest(score_func=chi2, k = 8)#this is feature selection

# Create the Model
Use a decision tree classifier as the final estimator in the pipeline.

In [44]:
transformer5  = DecisionTreeClassifier()

# Build the Pipeline
Chain preprocessing, feature selection, and the classifier into one reusable pipeline.

In [45]:
pipe = make_pipeline(transformer1,transformer2,transformer3,transformer4,transformer5)#now this will make the 
#pipeline

# Train the Pipeline
Fit the complete pipeline on the training data.

In [46]:
pipe.fit(X_train , Y_train)

,steps,"[('columntransformer-1', ...), ('columntransformer-2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Pipeline Note
Mark this section as the complete end-to-end pipeline.

In [47]:
#this is the whole pipeline

# Inspect Pipeline Steps
Display the named steps inside the fitted pipeline.

In [49]:
pipe.named_steps["columntransformer-2"]

,transformers,"[('ohe_sex_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,False
